# ⚖️ Judisurely — AI Legal Action Engine
**Track 1: AI for Legal Assistance | Build with Gemma – AIMS DTU**

### Before running
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. **Add Input → Models → Gemma 4 → gemma-4-e2b-it** (or gemma-4-e2b)
4. **Run → Restart Session** before a clean run

In [ ]:
import os, shutil, sys

os.chdir("/kaggle/working")
if os.path.exists("/kaggle/working/Judisurely"):
    shutil.rmtree("/kaggle/working/Judisurely")

!git clone https://github.com/Harsh7t/Judisurely.git /kaggle/working/Judisurely

os.chdir("/kaggle/working/Judisurely")
os.environ["NYAY_MITRA_ROOT"] = "/kaggle/working/Judisurely"

# Remove stale nested clone + cached wrong imports
nested = "/kaggle/working/Judisurely/Judisurely"
if os.path.exists(nested):
    shutil.rmtree(nested)

for m in list(sys.modules):
    if m == "utils" or m.startswith("utils."):
        del sys.modules[m]

sys.path = [p for p in sys.path if "Judisurely" not in p]
sys.path.insert(0, "/kaggle/working/Judisurely")

assert os.path.isfile("data/legal_kb.json")
print("Ready:", os.getcwd())

In [ ]:
# CRITICAL: Gemma 4 needs latest transformers (model_type=gemma4)
!pip install -q -U "git+https://github.com/huggingface/transformers.git"

# Step A: other pipeline deps
!pip install -q -r requirements-kaggle.txt

# Step B: Gradio 4.44 + pinned deps (keeps Gemma-compatible huggingface_hub)
!pip install -q "gradio==4.44.1" "gradio-client==1.3.0" --no-deps
!pip install -q aiofiles ffmpy python-multipart httpx orjson semantic-version toml typer "websockets==12.0" fastapi uvicorn starlette packaging anyio sniffio h11 jinja2
!pip install -q "importlib-resources<7.0" "markupsafe~=2.0" "Pillow<11.0" "tomlkit==0.12.0"

import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers:", transformers.__version__)
print("gemma4 supported:", "gemma4" in CONFIG_MAPPING)
assert "gemma4" in CONFIG_MAPPING, "gemma4 missing — Restart Session and re-run Cell 2"
print("Install done")

In [ ]:
import glob, os

os.chdir("/kaggle/working/Judisurely")

configs = glob.glob("/kaggle/input/models/**/config.json", recursive=True)
for c in configs:
    print(c)

assert configs, "❌ No model — Add Input → Models → gemma-4-e2b-it"
gemma_path = configs[0].replace("/config.json", "")
assert "gemma-4" in gemma_path, f"Wrong model: {gemma_path}"
print("✅ Gemma path:", gemma_path)

In [ ]:
import os, sys

os.chdir("/kaggle/working/Judisurely")
os.environ["NYAY_MITRA_ROOT"] = "/kaggle/working/Judisurely"

for m in list(sys.modules):
    if m == "utils" or m.startswith("utils."):
        del sys.modules[m]

sys.path = [p for p in sys.path if "Judisurely" not in p]
sys.path.insert(0, "/kaggle/working/Judisurely")

from utils.gemma_client import load_gemma
from utils.rag import load_knowledge_base
from utils.pipeline import analyze_notice
import utils.rag as rag_mod

print("rag loaded from:", rag_mod.__file__)
assert "Judisurely/Judisurely" not in rag_mod.__file__, "Wrong nested import path! Re-run Cell 1"

load_knowledge_base()
load_gemma()

sample = open("data/sample_notices/rent_notice_delhi.txt").read()
r = analyze_notice(text=sample, language="English")
print("Urgency:", r["reasoning"]["urgency_level"])
print("Summary:", r["reasoning"]["plain_language_summary"])

In [ ]:
import os
os.chdir("/kaggle/working/Judisurely")
!python run_kaggle.py